In [9]:
import pandas as pd
import geopandas as gpd
import os
from dotenv import load_dotenv

In [10]:
load_dotenv()

True

In [11]:
sheet_id = os.getenv("GOOGLE_SHEET_ID")

In [12]:
tabs = {
    "bayview": "0",
    # "excelsior": "1342562143",
    "tenderloin": "1609009848",
    # "chinatown": "89135915",
    "richmond": "388800952",
    "sunset": "1250830556",
    "mission": "1474907138",
}

dfs = {}

for name, gid in tabs.items():
    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
    dfs[name] = pd.read_csv(url)

In [13]:
dfs["bayview"].head()

,Item,Category,Type,Address,Phone,About,Hours,Website,Latitude,Longitude
0,Dr. Charles R. Drew College Preparatory Academy,School,Elementary,"50 Pomona St, San Francisco, CA 94124",(415) 330-1526,NaN,NaN,NaN,37.731756,-122.393896
1,Dr. George Washington Carver Elementary School,School,Elementary,"1360 Oakdale Ave, San Francisco, CA 94124",(415) 330-1540,NaN,NaN,NaN,37.731836,-122.385408
2,Bret Harte Elementary School,School,Elementary,"1035 Gilman Ave, San Francisco, CA 94124",(415) 330-1520,NaN,NaN,NaN,37.718505,-122.389148
3,E.R. Taylor Elementary School,School,Elementary,"423 Burrows St, San Francisco, CA 94134",(415) 330-1530,NaN,NaN,NaN,37.727439,-122.407100
4,Malcom X Academy Elementary School,School,Elementary,"200-398 Harbor Rd, San Francisco, CA 94124",(415) 695-5950,NaN,NaN,NaN,37.734579,-122.381061


In [14]:
for name, df in dfs.items():
    if df["About"].isna().any() or df["Hours"].isna().any() or df["Website"].isna().any():
        # replace NaN with empty string
        df["About"] = df["About"].fillna("")
        df["Hours"] = df["Hours"].fillna("")
        df["Website"] = df["Website"].fillna("")
        # drop Type column
        df.drop(columns=["Type"], inplace=True)

In [15]:
# # for each item in dfs, convert to geodf and save as geojson
# for name, df in dfs.items():
#     gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.Longitude, df.Latitude))
#     print(gdf.head())
#     gdf.to_file(f"../data/{name}.geojson", driver="GeoJSON")

In [16]:
base_dir = "../docs"

for name, df in dfs.items():
    # create ../docs/NAME-resources
    folder_name = f"{name}-resources"
    out_dir = os.path.join(base_dir, folder_name)
    os.makedirs(out_dir, exist_ok=True)

    # build GeoDataFrame
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
        crs="EPSG:4326"
    )

    print(name, gdf.head())

    # save as ../docs/NAME-resources/NAME.geojson
    out_path = os.path.join(out_dir, f"{name}.geojson")
    gdf.to_file(out_path, driver="GeoJSON")

bayview                                               Item Category  \
0  Dr. Charles R. Drew College Preparatory Academy   School   
1   Dr. George Washington Carver Elementary School   School   
2                     Bret Harte Elementary School   School   
3                    E.R. Taylor Elementary School   School   
4               Malcom X Academy Elementary School   School   

                                      Address           Phone About Hours  \
0       50 Pomona St, San Francisco, CA 94124  (415) 330-1526               
1   1360 Oakdale Ave, San Francisco, CA 94124  (415) 330-1540               
2    1035 Gilman Ave, San Francisco, CA 94124  (415) 330-1520               
3     423 Burrows St, San Francisco, CA 94134  (415) 330-1530               
4  200-398 Harbor Rd, San Francisco, CA 94124  (415) 695-5950               

  Website   Latitude   Longitude                     geometry  
0          37.731756 -122.393896  POINT (-122.39390 37.73176)  
1          37.731836 -